# One-Class SVM — Anomali Tespiti\nSadece normal veriyle eğitilir. Normal bölgenin dışındakiler anomali olarak işaretlenir.

In [ ]:
import numpy as np, matplotlib.pyplot as plt\nfrom sklearn.svm import OneClassSVM\nfrom sklearn.preprocessing import StandardScaler\nimport os; os.makedirs('cikti', exist_ok=True)

## Sentetik Veri ve Eğitim

In [ ]:
rng = np.random.RandomState(42)\nX_normal = 0.3 * rng.randn(200, 2)\nX_normal = np.vstack([X_normal, 0.3 * rng.randn(200, 2) + [2, 2]])\nX_outliers = rng.uniform(low=-3, high=5, size=(30, 2))\nX = np.vstack([X_normal, X_outliers])\nX = StandardScaler().fit_transform(X)\n\n# nu=0.05: üst sınır olarak %5 anomali bekle\nmodel = OneClassSVM(kernel='rbf', gamma=0.5, nu=0.05)\nmodel.fit(X_normal)\npreds = model.predict(X)\n\nnormal_idx = preds == 1; outlier_idx = preds == -1\nprint(f'Normal: {normal_idx.sum()}, Anomali: {outlier_idx.sum()}')

In [ ]:
xx, yy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))\nZ = model.decision_function(np.c_[xx.ravel(), yy.ravel()])\nZ = Z.reshape(xx.shape)\n\nplt.figure(figsize=(8, 6))\nplt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap='Blues_r', alpha=0.3)\nplt.contour(xx, yy, Z, levels=[0], colors='red', linewidths=2, linestyles='--')\nplt.scatter(X[normal_idx, 0], X[normal_idx, 1], c='blue', alpha=0.5, label='Normal')\nplt.scatter(X[outlier_idx, 0], X[outlier_idx, 1], c='red', alpha=0.7, label='Anomali', edgecolors='black')\nplt.title('One-Class SVM (gamma=0.5, nu=0.05)')\nplt.legend(); plt.savefig('cikti/one_class_svm.png', dpi=100, bbox_inches='tight')\nplt.show()

## Gamma Parametresi Etkisi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))\nfor ax, g in zip(axes, [0.1, 0.5, 2.0]):\n    m = OneClassSVM(kernel='rbf', gamma=g, nu=0.05)\n    m.fit(X_normal)\n    Z = m.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)\n    ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap='Blues_r', alpha=0.3)\n    ax.contour(xx, yy, Z, levels=[0], colors='red', linewidths=2, linestyles='--')\n    ax.scatter(X[:, 0], X[:, 1], c=preds, cmap='coolwarm', alpha=0.4, s=10)\n    ax.set_title(f'gamma={g}')\nplt.tight_layout(); plt.savefig('cikti/one_class_svm_gamma.png', dpi=100)\nplt.show()